In [ ]:
%pip install google-generativeai pandas openpyxl tqdm --quiet


In [ ]:
import os, json, time
import pandas as pd
from tqdm import tqdm
import google.generativeai as genai

print("Imports OK.")


In [ ]:
GEMINI_API_KEY = "YOUR_GEMINI_API_KEY"

INPUT_FILE = "DDL_TE_Scores.xlsx"
CHECKPOINT_CSV = "DDL_TRAI_Checkpoint.csv"
OUTPUT_EXCEL = "DDL_TRAI_Scores.xlsx"

GEMINI_MODEL = "gemini-2.5-flash-lite"

MAX_RETRIES = 3
DELAY = 0.3

genai.configure(api_key=GEMINI_API_KEY)
gemini_model = genai.GenerativeModel(GEMINI_MODEL)

STAGE_NAMES = {
    1: "Stage 1 — Discovery & Development",
    2: "Stage 2 — Preclinical Research",
    3: "Stage 3 — Clinical Research",
    4: "Stage 4 — Regulatory Review",
}

print(f"Gemini: {GEMINI_MODEL}")
print("Pipeline: Step 1 = Gemini synthesis | Step 2 = Gemini TRAI scoring")


In [ ]:
SYNTHESIS_SYSTEM = (
    "You are an expert in Generative AI capabilities and pharmaceutical drug development. "
    "Your task is to rewrite one Gemini AI exposure assessment of the same occupational task "
    "into a single consolidated motivation paragraph.\n\n"
    "You will receive one Gemini motivation explaining why a given DDL task received a "
    "particular AI capability rating (TE score). Rewrite it into one coherent paragraph "
    "(4-6 sentences) that summarises the key rationale and clearly states which Generative AI "
    "capabilities (LLM reasoning, Image Processing) are or are not applicable to this task, and why.\n\n"
    "Respond ONLY with the consolidated paragraph — no JSON, no headers, no preamble."
)

def build_synthesis_prompt(occupation, stage_name, task, mot_g):
    return (
        f'Occupation: "{occupation}"\n'
        f'DDL Stage: {stage_name}\n'
        f'Task: "{task}"\n\n'
        f'Gemini motivation:\n{mot_g}\n\n'
        f'Consolidated motivation:'
    )

TRAI_SCALE = (
    "Rate the level of AI substitutability on the following scale:\n"
    "  1 = No engagement — AI has no meaningful role; the task is entirely human-executed.\n"
    "  2 = Minimal engagement — AI can assist with minor peripheral elements only; core task execution requires full human involvement.\n"
    "  3 = Partial engagement — AI can perform a significant portion of the task, but human execution, judgement, or accountability remains central and irreplaceable.\n"
    "  4 = Substantial engagement — AI can perform most of the task with limited human oversight; human involvement is needed primarily for final validation or sign-off.\n"
    "  5 = Complete automation — AI can fully replace human execution; the task is entirely cognitive/information-based with no irreplaceable human element.\n"
)

TRAI_DISTINCTION = (
    "IMPORTANT — Capability vs. Substitutability:\n"
    "  - A task may be technically feasible for AI (high TE score) but not fully substitutable if regulatory frameworks require a named human responsible person, ethical or legal requirements mandate human presence or accountability, the task involves irreplaceable human judgement in high-stakes contexts, or the task requires physical presence that is not compensated by remote AI.\n"
    "  - Score based on actual replaceability of the human, not just technical feasibility.\n"
)

TRAI_FEW_SHOT = (
    "Five-shot examples:\n\n"
    "Occupation: Biostatisticians | DDL Stage: Stage 3 — Clinical Research\n"
    "Task: Calculate sample size requirements for clinical studies.\n"
    "Consolidated motivation: The assessment indicates this task is entirely cognitive and computational. LLMs can receive input parameters, select appropriate statistical frameworks, and produce documented sample size estimates with code. No physical component exists, and no human judgement irreplaceable by AI is required beyond parameter specification.\n"
    "{\"trai_score\": 5, \"trai_motivation\": \"This task is completely tractable by GenAI. Sample size calculation requires only computational reasoning and statistical knowledge, both of which are within current LLM capabilities. There is no physical execution component or required human interaction.\"}\n\n"
    "Occupation: Regulatory Affairs Specialists | DDL Stage: Stage 4 — Regulatory Review\n"
    "Task: Prepare and submit NDA dossier sections to the FDA.\n"
    "Consolidated motivation: The assessment indicates LLMs can draft, structure, and cross-reference dossier content at near-expert level. However, regulatory submission carries legal liability and requires a named qualified person under FDA/EMA frameworks.\n"
    "{\"trai_score\": 4, \"trai_motivation\": \"GenAI can perform the overwhelming cognitive core of NDA dossier preparation. The residual human requirement is legal accountability, which prevents complete automation but leaves substantial AI substitutability.\"}\n\n"
    "Occupation: Clinical Research Coordinators | DDL Stage: Stage 3 — Clinical Research\n"
    "Task: Recruit patients for clinical trials by screening records and contacting participants.\n"
    "Consolidated motivation: The assessment indicates AI can automate eligibility screening and draft recruitment communications, but direct patient contact for informed consent and trust-building requires human interaction.\n"
    "{\"trai_score\": 3, \"trai_motivation\": \"Recruitment combines automatable record screening with human consent and communication duties. Since the legally and ethically sensitive interaction remains human, AI can only partially substitute the task.\"}\n\n"
    "Occupation: Quality Control Systems Managers | DDL Stage: Stage 4 — Regulatory Review\n"
    "Task: Audit and inspect subcontractor facilities for GMP compliance.\n"
    "Consolidated motivation: The assessment indicates facility audits are fundamentally physical activities. AI can support preparation and reporting, but on-site inspection and real-time observation require human physical presence.\n"
    "{\"trai_score\": 2, \"trai_motivation\": \"AI can assist with peripheral audit preparation and report drafting, but cannot replace the core on-site inspection activity. The task therefore has minimal substitutability.\"}\n\n"
    "Occupation: Veterinary Assistants and Laboratory Animal Caretakers | DDL Stage: Stage 2 — Preclinical Research\n"
    "Task: Anesthetize and inoculate animals according to study protocol instructions.\n"
    "Consolidated motivation: The assessment indicates this task is a physical in-vivo laboratory procedure. Administering anaesthesia and injections requires direct manual interaction with live animals.\n"
    "{\"trai_score\": 1, \"trai_motivation\": \"This task is a physical in-vivo procedure with no cognitive core that GenAI could substitute. Any AI support is negligible relative to the manual execution requirement.\"}"
)

TRAI_SYSTEM = (
    "You are an expert in Generative AI capabilities, pharmaceutical drug development, and the future of work. "
    "Assess the degree to which Generative AI can replace or substitute human execution of a specific DDL occupational task.\n\n"
    "You will receive the occupation, DDL stage, task description, and a consolidated motivation summarising the Gemini AI capability assessment.\n\n"
    + TRAI_SCALE + "\n"
    + TRAI_DISTINCTION + "\n"
    + TRAI_FEW_SHOT + "\n\n"
    "Respond ONLY with valid JSON: "
    "{\"trai_score\": <integer 1-5>, \"trai_motivation\": \"<paragraph explaining the substitutability rating, explicitly distinguishing capability from replaceability>\"}"
)

def build_trai_prompt(occupation, stage_name, task, consolidated_motivation):
    return (
        f'Occupation: "{occupation}"\n'
        f'DDL Stage: {stage_name}\n'
        f'Task: "{task}"\n\n'
        f'Consolidated motivation:\n{consolidated_motivation}\n\n'
        f'Assign the TRAI score:'
    )

print(f"Synthesis prompt : {len(SYNTHESIS_SYSTEM):,} chars")
print(f"TRAI prompt      : {len(TRAI_SYSTEM):,} chars")
print("Prompts ready.")


In [ ]:
def parse_trai(raw):
    try:
        clean = raw.strip()
        if "```" in clean:
            parts = clean.split("```")
            clean = parts[1] if len(parts) > 1 else parts[0]
            if clean.startswith("json"):
                clean = clean[4:]
        start = clean.find("{")
        end = clean.rfind("}") + 1
        if start != -1 and end > start:
            clean = clean[start:end]
        parsed = json.loads(clean.strip())
        score = int(parsed.get("trai_score", -1))
        if score not in (1, 2, 3, 4, 5):
            raise ValueError(f"Invalid score: {score}")
        motivation = str(parsed.get("trai_motivation", "")).strip()
        if not motivation:
            raise ValueError("Empty motivation")
        return {"trai_score": score, "trai_motivation": motivation}
    except Exception as e:
        return {"trai_score": -1, "trai_motivation": f"Parse error: {str(e)[:120]}"}


def synthesise_motivations(occupation, stage_name, task, mot_g):
    prompt = build_synthesis_prompt(occupation, stage_name, task, mot_g)
    full = SYNTHESIS_SYSTEM + "\n\n" + prompt
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            resp = gemini_model.generate_content(
                full,
                generation_config=genai.types.GenerationConfig(temperature=0, max_output_tokens=400)
            )
            text = resp.text.strip()
            if text:
                return text
        except Exception as e:
            if "429" in str(e) or "quota" in str(e).lower():
                time.sleep(60 + attempt * 15)
            elif attempt < MAX_RETRIES:
                time.sleep(10)
    return "Synthesis failed — motivation unavailable."


def query_gemini_trai(occupation, stage_name, task, consolidated):
    prompt = build_trai_prompt(occupation, stage_name, task, consolidated)
    full = TRAI_SYSTEM + "\n\n" + prompt
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            resp = gemini_model.generate_content(
                full,
                generation_config=genai.types.GenerationConfig(temperature=0, max_output_tokens=500)
            )
            result = parse_trai(resp.text)
            if result["trai_score"] != -1:
                return result
            if attempt < MAX_RETRIES:
                time.sleep(10)
        except Exception as e:
            if "429" in str(e) or "quota" in str(e).lower():
                time.sleep(60 + attempt * 15)
            elif attempt < MAX_RETRIES:
                time.sleep(10)
    return {"trai_score": -1, "trai_motivation": "Gemini TRAI: all attempts failed"}

print("Gemini query functions ready.")


In [ ]:
smoke = [
    {
        "occ": "Biostatisticians",
        "stage": "Stage 3 — Clinical Research",
        "task": "Calculate sample size requirements for clinical studies.",
        "mot_g": "This task is entirely computational. LLMs can perform power calculations and generate documented code. No physical component or irreplaceable human judgement exists.",
    },
    {
        "occ": "Veterinary Assistants and Laboratory Animal Caretakers",
        "stage": "Stage 2 — Preclinical Research",
        "task": "Anesthetize and inoculate animals according to study protocol instructions.",
        "mot_g": "Physical in-vivo procedure. GenAI cannot perform any physical laboratory operations. Contribution is negligible.",
    },
]

print("=" * 72)
for s in smoke:
    print(f"\nTask : {s['task']}")
    print(f"Occ  : {s['occ']} | {s['stage']}")
    print("  → Step 1: Synthesising motivation with Gemini...")
    consolidated = synthesise_motivations(s["occ"], s["stage"], s["task"], s["mot_g"])
    print(f"  Consolidated: {consolidated[:160]}...")
    time.sleep(1)
    print("  → Step 2: TRAI scoring with Gemini...")
    g = query_gemini_trai(s["occ"], s["stage"], s["task"], consolidated)
    print(f"  Gemini [{g['trai_score']}]: {g['trai_motivation'][:90]}...")
    print(f"  → TRAI Score: {g['trai_score']}")
    print("-" * 72)

print("\nSmoke test complete.")


In [ ]:
df = pd.read_excel(INPUT_FILE, sheet_name="Task_TE_Scores")
df["Task ID"] = df["Task ID"].astype(str)

required = ["Task ID", "Title", "Task", "dominant_stage", "stage_label", "te_score", "motivation_gemini"]
missing = [c for c in required if c not in df.columns]
if missing:
    print(f"WARNING — missing columns: {missing}")
else:
    print("All required columns present.")

if "motivation_gemini" in df.columns:
    df["motivation_gemini"] = df["motivation_gemini"].fillna("No motivation available.")

print(f"\nLoaded: {len(df):,} tasks from {df['Title'].nunique()} occupations")
for s, n in STAGE_NAMES.items():
    c = (df["dominant_stage"] == s).sum()
    print(f"  {n}: {c} tasks")


In [ ]:
if os.path.exists(CHECKPOINT_CSV):
    done_df = pd.read_csv(CHECKPOINT_CSV, dtype={"Task ID": str})
    done_ids = set(done_df["Task ID"].tolist())
    print(f"Resume: {len(done_ids):,} tasks already processed.")
else:
    done_df = pd.DataFrame()
    done_ids = set()
    print("Starting fresh.")

remaining = df[~df["Task ID"].isin(done_ids)].copy()
print(f"Remaining: {len(remaining):,} tasks\n")

buffer = []

for _, row in tqdm(remaining.iterrows(), total=len(remaining), desc="TRAI Scoring"):
    task_id = str(row["Task ID"])
    occ = str(row["Title"])
    task_text = str(row["Task"])
    stage_lbl = str(row.get("stage_label", STAGE_NAMES.get(row.get("dominant_stage"), "")))
    mot_g = str(row.get("motivation_gemini", "No motivation available."))

    consolidated = synthesise_motivations(occ, stage_lbl, task_text, mot_g)
    time.sleep(DELAY)

    g = query_gemini_trai(occ, stage_lbl, task_text, consolidated)
    trai_score = g["trai_score"]
    n_valid_models = 1 if trai_score != -1 else 0

    buffer.append({
        "Task ID": task_id,
        "consolidated_motivation": consolidated,
        "trai_gemini": g["trai_score"],
        "trai_motivation_gemini": g["trai_motivation"],
        "trai_score": trai_score,
        "n_valid_models": n_valid_models,
    })

    if len(buffer) % 25 == 0:
        snap = pd.concat([done_df, pd.DataFrame(buffer)], ignore_index=True)
        snap.drop_duplicates(subset="Task ID", keep="last").to_csv(CHECKPOINT_CSV, index=False)
        print(f"  Checkpoint: {len(snap):,} tasks saved.")

final = pd.concat([done_df, pd.DataFrame(buffer)], ignore_index=True) if buffer else done_df
final.drop_duplicates(subset="Task ID", keep="last").to_csv(CHECKPOINT_CSV, index=False)
print(f"\nDone! {len(final):,} tasks processed.")
print(f"Errors (trai_score = -1): {(final['trai_score'] == -1).sum()}")


In [ ]:
trai_df = pd.read_csv(CHECKPOINT_CSV, dtype={"Task ID": str})

print("Checkpoint columns:", list(trai_df.columns))

available = [c for c in [
    "Task ID", "consolidated_motivation",
    "trai_gemini", "trai_motivation_gemini",
    "trai_score", "n_valid_models"
] if c in trai_df.columns]

print("Using columns:", available)

merged = df.merge(trai_df[available], on="Task ID", how="left")


In [ ]:
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter
from openpyxl import load_workbook
import os

STAGE_COLORS = {1:"D6E4F0", 2:"D5F5E3", 3:"FEF9E7", 4:"FADBD8"}
SCORE_COLORS = {5:"1A7A4A", 4:"52BE80", 3:"F0B429", 2:"E8735A", 1:"C0392B"}

if os.path.exists(OUTPUT_EXCEL):
    os.remove(OUTPUT_EXCEL)
    print(f"Deleted old (corrupted) {OUTPUT_EXCEL}")

try:
    with pd.ExcelWriter(OUTPUT_EXCEL, engine="openpyxl") as writer:
        task_out.to_excel(writer,  sheet_name="Task_TRAI_Scores",   index=False)
        stage_out.to_excel(writer, sheet_name="Stage_TRAI_Summary", index=False)
    print(f"Written successfully: {OUTPUT_EXCEL}")
except Exception as e:
    print(f"Write failed: {e}")
    raise

def style_ws(ws, score_cols):
    headers = [c.value for c in ws[1]]
    for cell in ws[1]:
        cell.font      = Font(bold=True, color="FFFFFF", name="Arial", size=10)
        cell.fill      = PatternFill("solid", fgColor="2E4057")
        cell.alignment = Alignment(horizontal="center", wrap_text=True)
    s_idx   = headers.index("dominant_stage") + 1 if "dominant_stage" in headers else None
    sc_idxs = [headers.index(c) + 1 for c in score_cols if c in headers]
    for row in ws.iter_rows(min_row=2):
        sv = row[s_idx-1].value if s_idx else None
        bg = STAGE_COLORS.get(sv, "FFFFFF")
        for cell in row:
            cell.fill      = PatternFill("solid", fgColor=bg)
            cell.font      = Font(name="Arial", size=9)
            cell.alignment = Alignment(wrap_text=True, vertical="top")
        for idx in sc_idxs:
            sc = row[idx-1]
            rv = round(sc.value) if sc.value and isinstance(sc.value, (int, float)) and sc.value > 0 else None
            if rv and rv in SCORE_COLORS:
                sc.fill = PatternFill("solid", fgColor=SCORE_COLORS[rv])
                sc.font = Font(name="Arial", size=9, bold=True, color="FFFFFF")
    for i in range(1, ws.max_column + 1):
        h = str(ws.cell(1, i).value or "")
        if h == "Task":                        ws.column_dimensions[get_column_letter(i)].width = 50
        elif "motivation" in h.lower():        ws.column_dimensions[get_column_letter(i)].width = 55
        elif h in ("Title", "stage_label"):    ws.column_dimensions[get_column_letter(i)].width = 28
        elif h in ("te_score", "trai_score"):  ws.column_dimensions[get_column_letter(i)].width = 11
        elif h.startswith(("trai_", "te_")):   ws.column_dimensions[get_column_letter(i)].width = 13
        else:                                  ws.column_dimensions[get_column_letter(i)].width = 14
    ws.freeze_panes = "A2"

try:
    wb = load_workbook(OUTPUT_EXCEL)
    style_ws(wb["Task_TRAI_Scores"],   ["te_score","trai_score","trai_gemini"])
    style_ws(wb["Stage_TRAI_Summary"], ["te_mean","trai_mean"])
    wb.save(OUTPUT_EXCEL)
    print("Styling applied.")
except Exception as e:
    print(f"Styling failed (file still usable): {e}")

print(f"\nDone: {OUTPUT_EXCEL}")
print(f"  Task_TRAI_Scores   : {len(task_out):,} rows")
print(f"  Stage_TRAI_Summary : {len(stage_out)} rows")